# COVID-19 Literature: A Time-Series Case Study

This notebook is a focused case study of how the pandemic reshaped biomedical publishing. It traces coronavirus-related research over time, measures the scale of the 2020 surge, and places it against the pre-pandemic baseline.

One methodological point governs the whole notebook. "COVID-19" and "SARS-CoV-2" as MeSH terms exist only from 2020, so counting them alone would make coronavirus research appear to spring from nothing. Coronavirus research existed before 2020 (SARS in 2003, MERS from 2012), indexed under older terms. To get an honest baseline, the modern terms are unioned with the older coronavirus descriptors. Counts are reported in both absolute and per-1,000 form, because the 2020-2021 period combines a genuine surge with unusually rapid indexing, so neither measure alone tells the full story.

Runs on the published metadata (MeSH and keywords are included; no abstracts needed).

## Setup

In [ ]:
import os, glob, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING: pick ONE option (same pattern as earlier notebooks)
# =====================================================================

# ---- OPTION A: LOCAL (active) ----
ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(ROOT, "data", "2_clean")

# ---- OPTION B: KAGGLE (uncomment on Kaggle) ----
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input; attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])
# =====================================================================

df = pd.read_parquet(DATA_DIR, columns=["uid", "year", "mesh_descriptors", "keywords"])
df = df[df["year"] <= 2025].copy()
print(f"loaded {len(df):,} records")
assert len(df) == df["uid"].nunique(), "duplicate PMIDs; dedup did not run"
df[["year"]].head(3)

## 1. Define the coronavirus term set

The pandemic's literature must be identified by the union of modern and historical coronavirus descriptors, otherwise the pre-2020 baseline vanishes. The set below covers SARS-CoV-2 and COVID-19 (2020 onward), the original SARS virus (2003), MERS (2012 onward), and the general coronavirus descriptors used throughout. A record counts as coronavirus-related if any of its MeSH descriptors falls in this set.

In [ ]:
COVID_MESH = {
    # modern pandemic terms (2020 onward):
    "COVID-19", "SARS-CoV-2", "COVID-19 Vaccines", "COVID-19 Testing",
    "COVID-19 Drug Treatment", "SARS-CoV-2 variants", "Post-Acute COVID-19 Syndrome",
    # historical and general coronavirus terms (pre-2020 baseline):
    "Coronavirus Infections", "Coronavirus", "SARS Virus",
    "Severe Acute Respiratory Syndrome", "Betacoronavirus",
    "Middle East Respiratory Syndrome Coronavirus", "Coronavirus 229E, Human",
    "Coronavirus NL63, Human", "Coronavirus OC43, Human",
}

def is_covid(mesh):
    if not isinstance(mesh, (list, np.ndarray)):
        return False
    return bool(set(mesh) & COVID_MESH)

df["is_covid"] = df["mesh_descriptors"].map(is_covid)
print(f"coronavirus-related articles (all years, union of terms): {df['is_covid'].sum():,}")
print(f"share of corpus: {df['is_covid'].mean()*100:.2f}%")

## 2. The time series: absolute counts

Coronavirus-related articles per year, using the unioned term set. The pre-2020 baseline and the 2020 surge are both visible only because historical terms are included.

In [ ]:
articles_per_year = df.groupby("year").size()
covid_count = df[df["is_covid"]].groupby("year").size().reindex(articles_per_year.index, fill_value=0)

plt.figure(figsize=(13, 5))
sns.lineplot(x=covid_count.index, y=covid_count.values, marker="o", color="#d9534f")
plt.axvline(2020, color="grey", ls="--", lw=1, label="2020")
plt.title("Coronavirus-related articles per year (union of all coronavirus terms)")
plt.xlabel("year"); plt.ylabel("articles"); plt.legend()
plt.tight_layout(); plt.show()
print(covid_count.to_string())

**What this shows:** a small but real pre-2020 baseline (with bumps around the 2003 SARS outbreak and the 2012-2015 MERS period) is dwarfed by the 2020 surge. The jump is genuine, but its exact height is also affected by the unusually fast indexing of pandemic research in 2020-2021, so the absolute count should be read alongside the share view in section 3. Without the historical terms, the entire pre-2020 portion of this line would be zero.

## 3. The time series: share of all research

Absolute counts grow partly because the corpus grows. Expressing coronavirus articles as a share per 1,000 of all articles that year shows how much of total research attention the topic captured, which is the more comparable measure across years.

In [ ]:
covid_share = (covid_count / articles_per_year * 1000)

plt.figure(figsize=(13, 5))
sns.lineplot(x=covid_share.index, y=covid_share.values, marker="o", color="#e0853f")
plt.axvline(2020, color="grey", ls="--", lw=1, label="2020")
plt.title("Coronavirus-related articles per 1,000 of all articles that year")
plt.xlabel("year"); plt.ylabel("per 1,000 articles"); plt.legend()
plt.tight_layout(); plt.show()
print(covid_share.round(1).to_string())

**What this shows:** as a share of all research, the pandemic's footprint is stark: coronavirus work goes from a fraction of a percent to a substantial share of everything published, then settles at a still-elevated level. The share view controls for corpus growth, so the rise here reflects genuine reallocation of research attention, not merely more papers overall. The post-2021 plateau (rather than a return to baseline) indicates the topic became a permanent part of the literature, not a transient spike.

## 4. The scale of the surge

Quantifying the jump directly: the fold-increase from the pre-pandemic baseline to the peak pandemic year, in both absolute and share terms. The baseline is the mean of 2015-2019 (excluding the MERS years' tail), the peak is the maximum pandemic year.

In [ ]:
baseline_years = range(2015, 2020)
baseline_abs = covid_count.loc[covid_count.index.isin(baseline_years)].mean()
baseline_shr = covid_share.loc[covid_share.index.isin(baseline_years)].mean()

peak_year = covid_count.loc[covid_count.index >= 2020].idxmax()
peak_abs = covid_count.loc[peak_year]
peak_shr = covid_share.loc[peak_year]

print(f"pre-pandemic baseline (2015-2019 mean): {baseline_abs:,.0f} articles/year "
      f"({baseline_shr:.1f} per 1,000)")
print(f"peak pandemic year ({peak_year}):        {peak_abs:,.0f} articles "
      f"({peak_shr:.1f} per 1,000)")
print(f"\nfold-increase, absolute: {peak_abs/baseline_abs:,.0f}x")
print(f"fold-increase, share:    {peak_shr/baseline_shr:,.0f}x")
print(f"total coronavirus articles 2020-2025: "
      f"{covid_count.loc[covid_count.index >= 2020].sum():,}")

**What this shows:** the surge is large by any measure. The share-based fold-increase is the more honest figure because it removes corpus growth; the absolute figure is inflated both by the genuine surge and by the rapid indexing of pandemic papers. Reporting both, rather than the larger absolute number alone, avoids overstating the effect.

## 5. What the pandemic literature is about

Within coronavirus-related articles, the most common co-occurring MeSH descriptors (excluding the coronavirus terms themselves and generic check-tags) show the topical shape of the pandemic literature: which aspects of the disease drew research attention.

In [ ]:
GENERIC = {"Humans", "Female", "Male", "Adult", "Middle Aged", "Aged", "Animals",
           "Adolescent", "Child", "Young Adult", "Aged, 80 and over", "Retrospective Studies",
           "Risk Factors", "Pandemics"}

covid_df = df[df["is_covid"]]
companion = collections.Counter()
for mesh in covid_df["mesh_descriptors"]:
    if isinstance(mesh, (list, np.ndarray)):
        for d in mesh:
            if d not in COVID_MESH and d not in GENERIC:
                companion[d] += 1

top = pd.Series(dict(companion.most_common(20)))[::-1]
plt.figure(figsize=(10, 8))
sns.barplot(x=top.values, y=top.index, color="#d9534f")
plt.title("Most common topics in coronavirus-related articles (companion MeSH)")
plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

**What this shows:** the companion descriptors reveal what the pandemic literature studied: clinical aspects (pneumonia, respiratory features), public-health dimensions (vaccination, quarantine, public-health measures), and the populations and comorbidities most affected. This is the topical anatomy of the surge, not just its size.

## 6. How the focus shifted within the pandemic

The pandemic literature itself evolved. Comparing the early pandemic (2020-2021) with the later period (2022-2025) shows which sub-topics rose as the field matured, from acute clinical concerns toward vaccines, variants, and longer-term effects.

In [ ]:
def companion_rate(sub):
    n = len(sub)
    cnt = collections.Counter()
    for mesh in sub["mesh_descriptors"]:
        if isinstance(mesh, (list, np.ndarray)):
            for d in set(mesh):
                if d not in COVID_MESH and d not in GENERIC:
                    cnt[d] += 1
    return {k: v / n * 1000 for k, v in cnt.items()}, n

early = covid_df[covid_df["year"].isin([2020, 2021])]
late  = covid_df[covid_df["year"].isin([2022, 2023, 2024, 2025])]
r_early, n_e = companion_rate(early)
r_late,  n_l = companion_rate(late)

rose = []
for k, rl in r_late.items():
    re_ = r_early.get(k, 0)
    if rl >= 5 and rl > re_ * 1.5 and rl * n_l / 1000 >= 30:
        rose.append((k, re_, rl))
rose.sort(key=lambda x: -(x[2] - x[1]))

print(f"early pandemic: {n_e:,} articles (2020-2021)")
print(f"later pandemic: {n_l:,} articles (2022-2025)")
print(f"\ntopics that rose from early to later pandemic (per 1,000 within coronavirus articles):\n")
for k, e, l in rose[:15]:
    print(f"  {e:6.1f} -> {l:6.1f}   {k}")

**What this shows:** the internal focus of the pandemic literature shifted as the emergency phase passed. Topics tied to vaccines, variants, long-term effects, and mental-health and social dimensions tend to rise in the later period, while acute-care topics that dominated 2020-2021 give way. This is the research community following the disease through its phases.

## 7. Summary and caveats

### What this notebook produced
This notebook traced coronavirus-related research from 2003 to 2025 using a union of modern and historical coronavirus terms, measured the 2020 surge in both absolute and share terms, quantified the fold-increase over the pre-pandemic baseline, mapped the topical anatomy of the pandemic literature, and tracked how its internal focus shifted from the early to the later pandemic.

### Caveats

**The pre-2020 baseline depends on the term union.** Counting only "COVID-19" and "SARS-CoV-2" would erase all coronavirus research before 2020. The union with historical terms (SARS Virus, MERS, Coronavirus Infections, and others) restores a real baseline, but the boundary of "coronavirus-related" is a definitional choice; a broader or narrower term set would shift the baseline. The set is explicit in section 1.

**The 2020-2021 surge is inflated by rapid indexing.** Pandemic papers were indexed unusually quickly, so the absolute count for those years overstates the steady-state rate. The share-based figures are more comparable across time, which is why both are reported. The fold-increase should be read from the share view, not the absolute.

**Counts are MeSH-based, so they inherit MeSH indexing limits.** A coronavirus paper not yet assigned a coronavirus descriptor would be missed, and the post-2019 MeSH-depth decline (EDA 6c) means recent papers carry fewer descriptors on average, which could slightly undercount recent coronavirus tagging. Keyword-based identification would be an alternative cross-check.

**This is US-affiliated, human-subject literature only.** Like the rest of the corpus, the coronavirus counts reflect the filtered scope, not global pandemic research output.

---

## Project notebooks complete

This notebook completed the analysis arc: exploratory analysis, MeSH co-occurrence structure, co-authorship, keyword trends, and this pandemic case study. Each runs on the published metadata and documents its own assumptions and limits.

See `README.md` for the project overview and the published Kaggle dataset for the metadata release.